# NFL DATA WAREHOUSE — PROJECT CONTROL CENTER

## Objective

Build a comprehensive historical NFL data warehouse using
free and publicly accessible sources.

## Data Categories

1. NFL play-by-play
2. Player statistics
3. Player profiles
4. Rosters and depth charts
5. Injuries and availability
6. Coaching histories
7. Draft and combine
8. College football
9. Teams and franchises
10. Stadiums
11. Weather
12. Contracts and transactions
13. Fantasy football
14. Betting markets
15. Historical records

## Storage Architecture

Raw → Bronze → Silver → Gold

## Primary Database

DuckDB

## Main Analytical Objective

Create a historical player-game dataset for:

QB, RB, WR, TE, K

Use the resulting data to train neural networks
for player performance forecasting.

In [ ]:
%run ../config/notebook_setup.py

In [3]:
import os
import sys
import json
import time
import logging
import hashlib
import warnings

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import polars as pl
import numpy as np
import duckdb

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

print("NFL Data Warehouse initialized")
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)

NFL Data Warehouse initialized
Python: 3.10.13 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:24:38) [MSC v.1916 64 bit (AMD64)]
Pandas: 2.0.3
Polars: 2.0.0
DuckDB: 1.5.6


## Define Directories

In [4]:
PROJECT_ROOT = Path.cwd()

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
EXPORT_DIR = DATA_DIR / "exports"
LOG_DIR = DATA_DIR / "logs"

DATABASE_DIR = PROJECT_ROOT / "database"

REPORT_DIR = PROJECT_ROOT / "reports"

for directory in [
    RAW_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    EXPORT_DIR,
    LOG_DIR,
    DATABASE_DIR,
    REPORT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

DB_PATH = DATABASE_DIR / "nfl_warehouse.duckdb"

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)

Project root: C:\Users\Stephen\Desktop\Sports\FantasySports\NFL\NFL_Data_Warehouse
Database: C:\Users\Stephen\Desktop\Sports\FantasySports\NFL\NFL_Data_Warehouse\database\nfl_warehouse.duckdb


## Configure Database

In [5]:
con = duckdb.connect(str(DB_PATH))

schemas = [
    "raw",
    "bronze",
    "silver",
    "gold",
    "reference",
    "metadata"
]

for schema in schemas:

    con.execute(
        f"CREATE SCHEMA IF NOT EXISTS {schema}"
    )

con.close()

print("Database successfully initialized")

Database successfully initialized


## Display Project Status

In [6]:
def get_project_status():

    con = duckdb.connect(
        str(DB_PATH),
        read_only=True
    )

    tables = con.execute("""
        SELECT
            table_schema,
            table_name
        FROM information_schema.tables
        ORDER BY table_schema, table_name
    """).df()

    con.close()

    return tables


display(get_project_status())

,table_schema,table_name
